# 📦 Lector de Archivos Parquet
> Carga, inspecciona y explora cualquier archivo `.parquet` con visualizaciones enriquecidas.

---

In [ ]:
# ── Dependencias ──────────────────────────────────────────────────────────────
# !pip install pandas pyarrow plotly seaborn matplotlib

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'pandas  : {pd.__version__}')
print(f'pyarrow : {pa.__version__}')
print(f'plotly  : {px.__version__}')
print('✅ Dependencias listas')

## 1. 📂 Cargar archivo

In [ ]:
# ── Ajusta solo esta ruta ──────────────────────────────────────────────────────
RUTA_PARQUET = 'tu_archivo.parquet'
# ──────────────────────────────────────────────────────────────────────────────

df = pd.read_parquet(RUTA_PARQUET)
tabla_pa = pq.read_table(RUTA_PARQUET)
meta = pq.read_metadata(RUTA_PARQUET)

print(f'✅ Archivo cargado: {RUTA_PARQUET}')
print(f'   Filas    : {df.shape[0]:,}')
print(f'   Columnas : {df.shape[1]}')
print(f'   Memoria  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## 2. 👁️ Vista previa

In [ ]:
# Primeras y últimas filas
print(f'── Primeras 5 filas ──')
display(df.head())
print(f'── Últimas 5 filas ──')
display(df.tail())

## 3. 🗂️ Esquema y tipos de datos

In [ ]:
# Resumen de columnas con tipos pandas y pyarrow
tipos_df = pd.DataFrame({
    'Tipo Pandas': df.dtypes.astype(str),
    'Tipo PyArrow': {field.name: str(field.type) for field in tabla_pa.schema},
    'No Nulos': df.count(),
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos': df.nunique()
})
display(tipos_df)

In [ ]:
# Distribución visual de tipos de columna
conteo_tipos = df.dtypes.astype(str).value_counts().reset_index()
conteo_tipos.columns = ['Tipo', 'Cantidad']

fig = px.pie(
    conteo_tipos, values='Cantidad', names='Tipo',
    title='Distribución de tipos de columna',
    hole=0.4, color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textinfo='label+percent+value')
fig.show()

## 4. 📊 Estadísticas descriptivas

In [ ]:
# Numéricas
numericas = df.select_dtypes(include='number')
if not numericas.empty:
    print('── Columnas numéricas ──')
    display(numericas.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std']))

# Categóricas / texto
categoricas = df.select_dtypes(exclude='number')
if not categoricas.empty:
    print('── Columnas no numéricas ──')
    display(categoricas.describe().T)

## 5. 🕳️ Mapa de valores nulos

In [ ]:
nulos = df.isnull().sum()
cols_con_nulos = nulos[nulos > 0]

if cols_con_nulos.empty:
    print('✅ No hay valores nulos en el dataset')
else:
    pct = (cols_con_nulos / len(df) * 100).round(2)
    fig = px.bar(
        x=cols_con_nulos.index, y=pct.values,
        labels={'x': 'Columna', 'y': '% Nulos'},
        title=f'Columnas con valores nulos ({len(cols_con_nulos)} de {df.shape[1]})',
        color=pct.values, color_continuous_scale='Reds',
        text=pct.values
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

    # Heatmap de nulos (muestra hasta 10k filas)
    muestra = df[cols_con_nulos.index].isnull().head(min(len(df), 10_000))
    fig2 = px.imshow(
        muestra.T.astype(int),
        labels=dict(color='Es Nulo'),
        color_continuous_scale=['white', 'crimson'],
        aspect='auto',
        title='Heatmap de nulos (rojo = nulo)'
    )
    fig2.update_coloraxes(showscale=False)
    fig2.show()

## 6. 📈 Distribuciones numéricas

In [ ]:
numericas = df.select_dtypes(include='number')

if numericas.empty:
    print('⚠️ No hay columnas numéricas')
else:
    MAX_COLS = 12  # máximo de columnas a graficar
    cols = numericas.columns[:MAX_COLS]
    n = len(cols)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=list(cols))

    for i, col in enumerate(cols):
        r, c = divmod(i, ncols)
        datos = numericas[col].dropna()
        fig.add_trace(
            go.Histogram(x=datos, name=col, marker_color='steelblue',
                         nbinsx=30, showlegend=False),
            row=r+1, col=c+1
        )

    fig.update_layout(
        height=300*nrows, title_text='Histogramas — columnas numéricas',
        showlegend=False
    )
    fig.show()

## 7. 🔠 Columnas categóricas — top valores

In [ ]:
cats = df.select_dtypes(exclude='number')
MAX_CAT = 6   # columnas a mostrar
TOP_N   = 10  # valores top por columna

if cats.empty:
    print('⚠️ No hay columnas categóricas')
else:
    cols_cat = [c for c in cats.columns if df[c].nunique() <= 100][:MAX_CAT]
    if not cols_cat:
        print('⚠️ No hay columnas categóricas con ≤100 valores únicos para graficar')
    else:
        ncols = min(2, len(cols_cat))
        nrows = (len(cols_cat) + ncols - 1) // ncols
        fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols_cat)

        for i, col in enumerate(cols_cat):
            r, c = divmod(i, ncols)
            vc = df[col].value_counts().head(TOP_N)
            fig.add_trace(
                go.Bar(x=vc.values, y=vc.index.astype(str), orientation='h',
                       name=col, showlegend=False,
                       marker_color='mediumseagreen'),
                row=r+1, col=c+1
            )

        fig.update_layout(
            height=350*nrows,
            title_text=f'Top {TOP_N} valores — columnas categóricas'
        )
        fig.show()

## 8. 🔥 Correlaciones (columnas numéricas)

In [ ]:
numericas = df.select_dtypes(include='number')

if numericas.shape[1] < 2:
    print('⚠️ Se necesitan al menos 2 columnas numéricas para calcular correlaciones')
else:
    corr = numericas.corr()
    fig = px.imshow(
        corr,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        text_auto='.2f',
        title='Mapa de correlaciones (Pearson)',
        aspect='auto'
    )
    fig.update_layout(height=600)
    fig.show()

## 9. 🗄️ Metadatos del archivo Parquet

In [ ]:
print(f'Versión formato  : {meta.format_version}')
print(f'Filas totales    : {meta.num_rows:,}')
print(f'Columnas         : {meta.num_columns}')
print(f'Grupos de filas  : {meta.num_row_groups}')
print(f'Tamaño serial.   : {meta.serialized_size:,} bytes')
print()

rg_data = []
for i in range(meta.num_row_groups):
    rg = meta.row_group(i)
    rg_data.append({
        'Grupo': i,
        'Filas': rg.num_rows,
        'Bytes': rg.total_byte_size,
        'KB': round(rg.total_byte_size / 1024, 1)
    })

display(pd.DataFrame(rg_data))

## 10. 🔍 Lectura parcial / filtrada (eficiente para archivos grandes)

In [ ]:
# ── Descomenta y ajusta lo que necesites ──────────────────────────────────────

# Leer solo columnas específicas
# df_cols = pd.read_parquet(RUTA_PARQUET, columns=['col1', 'col2'])

# Leer con filtro de filas (pushdown → no carga todo en RAM)
# df_filtrado = pd.read_parquet(RUTA_PARQUET, filters=[('columna', '=', 'valor')])
# df_filtrado = pd.read_parquet(RUTA_PARQUET, filters=[('edad', '>', 30), ('ciudad', '==', 'Loja')])

# Leer un rango de filas con pyarrow
# tabla_rango = pq.read_table(RUTA_PARQUET).slice(0, 1000)  # primeras 1000 filas
# df_rango = tabla_rango.to_pandas()

print('Descomenta las celdas que necesites ↑')